# Step 3. Write, and read back

*With ngff-zarr.*

**Goal:** put the two images and the transformation between them in one OME-Zarr 0.6 file, and check what a reader
finds in it. No registration tool is needed from here on.

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

import ngff_zarr as nz
import numpy as np
from common import DATA, scene_transformation
from ngff_zarr import CoordinateSystemIdentifier, NgffScene

FIXED, MOVING = f"{DATA}/fixed.ome.zarr", f"{DATA}/moving.ome.zarr"
TRANSFORM = "../output/registration/P000/Transform.ome.zarr"
STORE, SCENE = "../output/pair.ome.zarr", "../output/pair.ozx"
if Path(STORE).exists():
    shutil.rmtree(STORE)  # an earlier run of this cell: replaced

# An affine goes into the scene's metadata; a displacements field is copied below the scene, and pointed at.
registration = scene_transformation(TRANSFORM, STORE)

# The scene names what it connects: from the coordinates of the fixed image to those of the moving one.
registration.input = CoordinateSystemIdentifier(path="fixed", name="intrinsic")
registration.output = CoordinateSystemIdentifier(path="moving", name="intrinsic")
images = {"fixed": nz.from_ome_zarr(FIXED), "moving": nz.from_ome_zarr(MOVING)}
NgffScene(images=images, coordinateTransformations=[registration]).to_ome_zarr(STORE, overwrite=False)

# One file: the same store, zipped (RFC-9).
nz.write_store_to_zip(STORE, SCENE, version="0.6")
print(f"{SCENE}: an RFC-5 '{registration.type}', {Path(SCENE).stat().st_size / 1e6:.0f} MB")

../output/pair.ozx: an RFC-5 'affine', 42 MB


`pair.ozx` is one file:

```text
pair.ozx
├── zarr.json                      the scene: the transformation, and what it connects
├── fixed/                         the fixed image, with its pyramid
├── moving/                        the moving image, with its pyramid
└── coordinateTransformations/     a displacements field, when the preset wrote one; nothing for an affine
    └── registration/
```

- **An affine is twelve numbers.** Step 2 wrote it as an RFC-5 `affine`, a 3 × 4 matrix over (z, y, x) in micrometers,
  and the scene carries it in its own metadata. A `displacements` field is an OME-Zarr image of vectors: it is copied
  below the scene, which points at it by its path. `scene_transformation` in [common.py](common.py) does the one or
  the other, and [write.py](write.py) is this cell as a script.
- **The scene copies the images, as they are.** RFC-5 names them by a path inside the scene, so `pair.ozx` holds the two
  pyramids: 42 MB, all of it the images, unchanged. What is never written is a *resampled* image: the transformation is
  applied when a reader asks, in step 4.
- **The scene says what it connects:** from the coordinate system of `fixed` to the one of `moving`.

## Read it back

This is what a reader finds: the file's metadata, and the transformation in full.

In [ ]:
scene = NgffScene.from_ome_zarr(SCENE)
for name, multiscales in scene.images.items():
    size = " x ".join(str(extent) for extent in multiscales.images[0].data.shape)
    print(f"{name}: {size} voxels, {len(multiscales.images)} levels")

transformation = scene.coordinateTransformations[0]
print(f"transformation: {transformation.type}, from {transformation.input.path} to {transformation.output.path}")
if transformation.type == "affine":
    print(np.array2string(np.asarray(transformation.affine), precision=3, suppress_small=True))
else:
    field = nz.from_ome_zarr(f"{SCENE}/{transformation.path}").images[0]
    print(f"field: {' x '.join(str(extent) for extent in field.data.shape)}, axes {field.dims}")

fixed: 50 x 680 x 535 voxels, 3 levels
moving: 50 x 680 x 535 voxels, 3 levels
transformation: affine, from fixed to moving
[[  0.999  -0.013   0.034   0.344]
 [  0.013   1.      0.001 -10.088]
 [ -0.035  -0.001   0.999  37.001]]


The matrix maps a point of the fixed image to the same point of the moving image, in micrometers and in the order
z, y, x: a rotation of a few degrees, and in the last column the translation, 37 µm along x.

A reader of OME-Zarr 0.6 finds the same thing in `zarr.json`, at the root of the file:

In [ ]:
with zipfile.ZipFile(SCENE) as archive:
    root = json.loads(archive.read("zarr.json"))
print(json.dumps(root["attributes"]["ome"]["scene"]["coordinateTransformations"], indent=2))

[
  {
    "input": {
      "path": "fixed",
      "name": "intrinsic"
    },
    "output": {
      "path": "moving",
      "name": "intrinsic"
    },
    "type": "affine",
    "affine": [
      [
        0.999316790401152,
        -0.013299107171981128,
        0.03448312878449811,
        0.3436142498125642
      ],
      [
        0.01325527158319062,
        0.9999110206388373,
        0.0014995266767758864,
        -10.088313475502503
      ],
      [
        -0.0345000028637088,
        -0.0010414189486809278,
        0.9994041551089197,
        37.00061501043516
      ]
    ]
  }
]


## The size of the transformation

| Preset | What step 2 writes | `Transform.ome.zarr` | `pair.ozx` |
|---|---|---|---|
| `Generic_Rigid` | an `affine`: twelve numbers | 4 kB | 42 MB, the two images |
| `Generic_Rigid_BSpline` | a `displacements` field: float32, on the fixed grid | 157 MB | 196 MB |

The ITK form the preset writes without `Transform:omezarr` tells the same story: an `AffineTransform` of a few
kilobytes, or a 437 MB field.

**Next: [Step 4. Apply](4-apply.ipynb)**